# Scraper de calidad del aire

In [1]:
from io import StringIO

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


SCRAPER_URL = "https://www.valencia.es/valenciaalminut/"
SCRAPER_TABLE_ID = "tabla_dinamica"
SCRAPER_COLUMNS = ["µg/m3", "SO2", "NO2", "O3", "PM-10", "PM-2.5"]


def scrape_current_table() -> pd.DataFrame:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-notifications")
    options.add_argument("--ignore-certificate-errors")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(SCRAPER_URL)
        table_html = WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.ID, SCRAPER_TABLE_ID))
        ).get_attribute("outerHTML")
    finally:
        driver.quit()

    df = pd.read_html(StringIO(table_html))[0]
    df = df[SCRAPER_COLUMNS].dropna(how="all")
    if df.empty or df["µg/m3"].dropna().empty:
        raise RuntimeError("Scrape returned no station rows.")
    return df

df = scrape_current_table()
print(df.to_string(index=False))

          µg/m3   SO2  NO2    O3  PM-10  PM-2.5
   AVDA.FRANCIA   1.0    7  86.0   24.0     8.0
   BULEVARD SUD   3.0   16  80.0    NaN     NaN
   MOLÍ DEL SOL 104.0   11  90.0   20.0    11.0
 PISTA DE SILLA   1.0   18  64.0   22.0    11.0
     POLITÈCNIC   1.0   12 101.0   15.0    11.0
         VIVERS   0.0    7 100.0    NaN     NaN
VALÈNCIA CENTRE   NaN    9   NaN   27.0    11.0
       DR.LLUCH   NaN   15   NaN   27.0    12.0
       CABANYAL   NaN    9   NaN   30.0    14.0
      OLIVERETA   NaN   28   NaN   31.0    15.0
        PATRAIX   NaN   58   NaN   28.0    13.0
